In [6]:
#1. Импорт библиотек#
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

pd.set_option('display.max_columns', 100)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')

In [7]:
#2. Загрузка данных#

In [8]:
DATA_DIR = Path('.')

guild_path = DATA_DIR / 'guild_data.csv'
energy_path = DATA_DIR / 'energy.csv'
payers_path = DATA_DIR / 'payers.csv'

print('guild_data.csv:', 'найден' if guild_path.exists() else 'ОТСУТСТВУЕТ')
print('energy.csv:', 'найден' if energy_path.exists() else 'ОТСУТСТВУЕТ')
print('payers.csv:', 'найден' if payers_path.exists() else 'ОТСУТСТВУЕТ')

payers = pd.read_csv(payers_path) if payers_path.exists() else None
if payers is not None:
    print('Размер payers:', payers.shape)
    display(payers.head())

guild_data.csv: ОТСУТСТВУЕТ
energy.csv: ОТСУТСТВУЕТ
payers.csv: найден
Размер payers: (20430, 2)


,revenue,character_id
0,102.90,13642352
1,105.84,29703528
2,50.04,28081545
3,18.04,12663426
4,5.91,16071934


In [ ]:
#3. Методика#

In [13]:
#Ячейка Code 7#

In [17]:
if not (guild_path.exists() and energy_path.exists()):
    print('АНАЛИЗ ГИЛЬДИЙ НЕ ЗАПУЩЕН: отсутствуют guild_data.csv и/или energy.csv.')
else:
    guilds = pd.read_csv(guild_path)
    energy = pd.read_csv(energy_path)
    print('guild_data:', guilds.shape)
    print('energy:', energy.shape)
    display(guilds.head())
    display(energy.head())

АНАЛИЗ ГИЛЬДИЙ НЕ ЗАПУЩЕН: отсутствуют guild_data.csv и/или energy.csv.


In [ ]:
#Ячейка Code 8#

In [18]:
if 'guilds' in globals() and 'energy' in globals():
    guilds['guild_created.datetime'] = pd.to_datetime(
        guilds['guild_created.datetime'], errors='coerce'
    )

    energy_total = (
        energy.groupby('guild_id', as_index=False)['energy_delta']
        .sum()
        .rename(columns={'energy_delta': 'energy_total'})
    )

    df = guilds.merge(
        energy_total,
        left_on='guild_created.guild_id',
        right_on='guild_id',
        how='left'
    )
    df['energy_total'] = df['energy_total'].fillna(0)

    df = df[
        (df['guild_created.datetime'].dt.year == 2022) &
        (df['guild_created.datetime'].dt.month.isin([11, 12]))
    ].copy()

    print('Гильдий после фильтрации:', len(df))

In [ ]:
#4. Подготовка признаков активности# #Ячейка Code 10 #

In [23]:
if 'df' in globals():
    feature_cols = [
        'n_added_characters', 'n_removed_characters', 'n_talents',
        'level', 'n_missions_completed', 'n_times_flag_turned_on',
        'mean_place', 'mean_rating', 'n_seasons', 'energy_total'
    ]

    X = df[feature_cols].replace([np.inf, -np.inf], np.nan)
    X = X.fillna(X.median(numeric_only=True))

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    display(X.describe().T)

In [24]:
#5. Выбор количества кластеров#

In [25]:
#Ячейка Code 12 #

In [26]:
if 'X_scaled' in globals():
    max_k = min(8, len(df) - 1)
    ks = range(2, max_k + 1)
    inertias, silhouettes = [], []

    for k in ks:
        model = KMeans(n_clusters=k, random_state=42, n_init=20)
        labels = model.fit_predict(X_scaled)
        inertias.append(model.inertia_)
        silhouettes.append(silhouette_score(X_scaled, labels))

    scores = pd.DataFrame({
        'k': list(ks),
        'inertia': inertias,
        'silhouette': silhouettes
    })
    display(scores)

    best_k = int(scores.loc[scores['silhouette'].idxmax(), 'k'])
    print('Оптимальное число кластеров:', best_k)

    plt.figure(figsize=(8, 5))
    plt.plot(scores['k'], scores['inertia'], marker='o')
    plt.title('Метод локтя')
    plt.xlabel('Количество кластеров')
    plt.ylabel('Inertia')
    plt.grid(alpha=.3)
    plt.show()

    plt.figure(figsize=(8, 5))
    plt.plot(scores['k'], scores['silhouette'], marker='o')
    plt.title('Silhouette score')
    plt.xlabel('Количество кластеров')
    plt.ylabel('Silhouette')
    plt.grid(alpha=.3)
    plt.show()

In [ ]:
#6. Кластеризация и профиль групп#

In [ ]:
#Ячейка Code 14#

In [27]:
if 'best_k' in globals():
    kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=20)
    df['cluster'] = kmeans.fit_predict(X_scaled)

    cluster_sizes = (
        df['cluster'].value_counts().sort_index().rename('guild_count').to_frame()
    )
    cluster_sizes['share'] = cluster_sizes['guild_count'] / len(df)
    display(cluster_sizes)

    profile = df.groupby('cluster')[feature_cols].mean().round(2)
    display(profile)

    relative_profile = (profile / df[feature_cols].mean()).round(2)
    display(relative_profile)

In [ ]:
#7. Интерпретация активности#

In [ ]:
#Ячейка Code 16#

In [28]:
if 'profile' in globals():
    activity_cols = [c for c in feature_cols if c != 'mean_place']
    cluster_index = (relative_profile[activity_cols].mean(axis=1)).sort_values()
    ordered = list(cluster_index.index)

    names = {ordered[0]: 'Низкая активность', ordered[-1]: 'Высокая активность'}
    for c in ordered[1:-1]:
        names[c] = 'Средняя активность'

    df['activity_group'] = df['cluster'].map(names)
    print('Соответствие кластеров:', names)
    display(df['activity_group'].value_counts().to_frame('guild_count'))

In [ ]:
#8. Визуализация#

In [30]:
#Ячейка Code 18#

In [31]:
if 'profile' in globals():
    profile.T.plot(kind='bar', figsize=(14, 6))
    plt.title('Средние значения признаков по кластерам')
    plt.xlabel('Признак')
    plt.ylabel('Среднее значение')
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

In [ ]:
#9. Гипотеза №1#
#Предположение: большинство созданных гильдий неактивны.#
#Критерий: если доля группы Низкая активность превышает 50%, гипотеза поддерживается выбранной методикой; иначе — не подтверждается.#

In [ ]:
#Ячейка Code 20#

In [34]:
if 'activity_group' in df.columns:
    shares = (df['activity_group'].value_counts(normalize=True) * 100).round(2)
    display(shares.to_frame('share_percent'))
    low_share = shares.get('Низкая активность', 0)
    print(f'Доля низкой активности: {low_share:.2f}%')
    print('Вывод:', 'гипотеза поддерживается' if low_share > 50 else 'гипотеза не подтверждается')

NameError: name 'df' is not defined

In [ ]:
#10. Paying Share и ARPPU#
#Создатель гильдии сопоставляется с payers.csv по creator_character_id = character_id.#
#Paying Share = платящие создатели / все создатели.#
#ARPPU = сумма платежей платящих создателей / число платящих создателей.#

#Ячейка Code 22#

In [38]:
if 'df' in globals() and payers is not None and 'activity_group' in df.columns:
    payers = payers.copy()
    payers['total_payments'] = pd.to_numeric(
        payers['total_payments'], errors='coerce'
    ).fillna(0)

    creator_payments = df[
        ['guild_created.guild_id', 'guild_created.creator_character_id', 'activity_group']
    ].merge(
        payers[['character_id', 'total_payments']],
        left_on='guild_created.creator_character_id',
        right_on='character_id',
        how='left'
    )

    creator_payments['total_payments'] = creator_payments['total_payments'].fillna(0)
    creator_payments['is_payer'] = creator_payments['total_payments'] > 0

    payment_stats = creator_payments.groupby('activity_group').agg(
        guilds=('guild_created.guild_id', 'nunique'),
        payers=('is_payer', 'sum'),
        total_revenue=('total_payments', 'sum')
    ).reset_index()

    payment_stats['paying_share'] = (
        payment_stats['payers'] / payment_stats['guilds']
    )
    payment_stats['ARPPU'] = np.where(
        payment_stats['payers'] > 0,
        payment_stats['total_revenue'] / payment_stats['payers'],
        0
    )

    display(payment_stats)

In [ ]:
#11. Гипотеза №2#
#Предположение: платящие игроки создают более активные гильдии.#
#Сравниваются Paying Share и ARPPU между группами активности. Более высокие показатели у активной группы будут означать наблюдаемую связь, 
но не доказывают причинность#

#Ячейка Code 24#

In [39]:
if 'payment_stats' in globals():
    display(payment_stats[
        ['activity_group', 'guilds', 'payers', 'paying_share', 'ARPPU']
    ])

    high = payment_stats[payment_stats['activity_group'] == 'Высокая активность']
    low = payment_stats[payment_stats['activity_group'] == 'Низкая активность']

    if not high.empty and not low.empty:
        print('Разница Paying Share:',
              f"{high['paying_share'].iloc[0] - low['paying_share'].iloc[0]:.2%}")
        print('Разница ARPPU:',
              f"${high['ARPPU'].iloc[0] - low['ARPPU'].iloc[0]:,.2f}")

In [ ]:
#12. Итоговый вывод#
#Гипотеза №1#
#Проверяется доля гильдий с низкой активностью относительно 50%.#

#Гипотеза №2#
#Сравниваются Paying Share и ARPPU групп высокой и низкой активности. Разница показывает статистическую связь, а не причинно-следственный эффект.#

#Гипотеза №3#
#Решение о существенном повышении стоимости создания гильдии нельзя делать только на основании наличия неактивных гильдий.#
#Необходимо учитывать результаты анализа активности и цель механики — вовлечение и удержание игроков.#
#В текущем комплекте без guild_data.csv и energy.csv итоговые численные выводы по этим гипотезам не формируются, 
чтобы не подменять реальные данные вымышленными.#